In [1]:
import os

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from astropy.coordinates import SkyCoord
from astropy.io import fits
from gammapy.irf import load_irf_dict_from_file
from scipy.stats import chi2, poisson
from scipy.optimize import curve_fit

**Adjust with the correct file paths**

In [2]:
name = "Mrk501"
blazar_coord = SkyCoord.from_name(name)
data_folder = "../cta_dc_data/mrk_501/"
irf_file = "../cta_dc_data/irfs/Prod5-North-40deg-AverageAz-4LSTs09MSTs.180000s-v0.1.fits.gz"

In [3]:
blazar_l = blazar_coord.galactic.l.deg
blazar_b = blazar_coord.galactic.b.deg

hbus = []
for filename in os.listdir(data_folder):
    hbus.append(fits.open(data_folder + filename))

irf = load_irf_dict_from_file(irf_file)
aeff = irf["aeff"]
print(f"Pointings: {len(hbus)}")

Pointings: 126


In [4]:
e_bins=np.logspace(np.log10(0.05), np.log10(50), 16)
e_mins=e_bins[:-1]
e_maxs=e_bins[1:]
e=np.sqrt(e_mins*e_maxs)
de=e_maxs-e_mins

th2_bins=np.linspace(0,0.16,17)
th2=(th2_bins[1:]+th2_bins[:-1])/2.

**Data collecting and overview**

In [5]:
bkg_subtraction_radius = 0.4

nbr_pointings = np.logspace(np.log10(10), np.log10(len(hbus)), 100)
t_expos = np.zeros_like(nbr_pointings)

cts_s = np.zeros((len(nbr_pointings), len(e), len(th2)))
cts_b = np.zeros((len(nbr_pointings), len(e), len(th2)))

for i, nbr in enumerate(nbr_pointings):
    nbr_now = 0
    for hbu in hbus:
        if nbr_now >= nbr:
            continue
        nbr_now += 1
        data_raw = hbu["EVENTS"].data

        t = data_raw["TIME"]
        t = np.sort(t)
        t_expos[i] += (t[-1] - t[0])/60/60

        coord = SkyCoord(
            ra=data_raw["RA"] * u.deg, dec=data_raw["DEC"] * u.deg, frame="icrs"
        )

        pointing_coord = SkyCoord(
            ra=hbu[1].header["RA_PNT"] * u.deg,
            dec=hbu[1].header["DEC_PNT"] * u.deg,
            frame="icrs",
        )
    
        bkg_center = SkyCoord(
            l=2 * pointing_coord.galactic.l - blazar_coord.galactic.l,
            b=2 * pointing_coord.galactic.b - blazar_coord.galactic.b,
            frame="galactic",
        )
        
        seps = coord.separation(blazar_coord).degree
        seps_b = coord.separation(bkg_center).degree
        src_pos_mask = seps < bkg_subtraction_radius
        bkg_pos_mask = seps_b < bkg_subtraction_radius
    
        energ = data_raw["ENERGY"]
    
        for j, (energ_min, energ_max) in enumerate(zip(e_mins, e_maxs)):
            m_s = (energ>energ_min) & (energ<energ_max) & src_pos_mask
            m_b = (energ>energ_min) & (energ<energ_max )& bkg_pos_mask
    
            h_s, _ = np.histogram(seps[m_s]**2, bins=th2_bins)
            h_b, _ = np.histogram(seps_b[m_b]**2, bins=th2_bins)
            
            cts_s[i][j] += h_s
            cts_b[i][j] += h_b

print(f"Exposure times = {t_expos}h")

Exposure times = [ 1.43073122  1.5735149   1.5735149   1.5735149   1.71724364  1.71724364
  1.71724364  1.71724364  1.86124769  1.86124769  1.86124769  1.94555313
  1.94555313  1.94555313  2.08979672  2.08979672  2.23368933  2.23368933
  2.23368933  2.37745217  2.37745217  2.51952364  2.51952364  2.66374885
  2.66374885  2.66374885  2.8079252   2.8079252   2.94993317  3.0937626
  3.0937626   3.23735429  3.23735429  3.37902355  3.37902355  3.5224833
  3.6647797   3.6647797   3.80720022  3.94970599  3.94970599  4.09358409
  4.23766798  4.37955823  4.37955823  4.52159753  4.66544779  4.80770185
  4.9500472   5.09206854  5.09206854  5.23259831  5.37660425  5.52061347
  5.66246224  5.80617993  5.94934545  6.2346488   6.37733947  6.52168365
  6.66543941  6.80955262  6.95079505  7.23865364  7.38287569  7.5252482
  7.81345021  7.9561841   8.09770783  8.38345434  8.52777478  8.81332783
  9.09785387  9.24201038  9.46912343  9.68450427  9.82867721 10.11411253
 10.39936732 10.68431432 10.97248808 

In [6]:
cts = np.zeros((len(nbr_pointings), len(e), len(th2)))
cts_err = np.zeros((len(nbr_pointings), len(e), len(th2)))

for i in range(len(nbr_pointings)):
    cts[i] = cts_s[i] - cts_b[i]
    cts_err[i] = np.sqrt(cts_s[i] + cts_b[i])

In [7]:
psf_hbu = irf["psf"]
print(psf_hbu)
print(psf_hbu.info())
print(psf_hbu.data.shape)
# here i'm guessing the energy bins of the psf since idk how to see them.
psf_e_bins = np.logspace(np.log10(0.013), np.log10(199.526), 22)
psf_e_means = np.sqrt(psf_e_bins[1:]*psf_e_bins[:-1])
print(psf_e_bins)

EnergyDependentMultiGaussPSF
----------------------------

  axes      : ['energy_true', 'offset']
  shape     : (21, 6)
  ndim      : 2
  parameters: ['sigma_1', 'sigma_2', 'sigma_3', 'scale', 'ampl_2', 'ampl_3']


Summary PSF info
----------------
Theta          : size =     6, min =  0.500 deg, max =  5.500 deg
Energy hi      : size =    21, min =  0.020 TeV, max = 199.526 TeV
Energy lo      : size =    21, min =  0.013 TeV, max = 125.893 TeV
68.00 containment radius at offset = 0.0 deg and energy_true =  1.0 TeV: 0.074 deg
95.00 containment radius at offset = 0.0 deg and energy_true =  1.0 TeV: 0.121 deg
68.00 containment radius at offset = 0.0 deg and energy_true = 10.0 TeV: 0.062 deg
95.00 containment radius at offset = 0.0 deg and energy_true = 10.0 TeV: 0.100 deg

(21, 6)
[1.30000000e-02 2.05721347e-02 3.25548251e-02 5.15170959e-02
 8.15243565e-02 1.29010003e-01 2.04154705e-01 3.23069084e-01
 5.11247747e-01 8.09035193e-01 1.28027546e+00 2.02599994e+00
 3.20608797e+00 5.07354412

In [42]:
def calculate_timedep_for_given_energ(energ_index):
    # Vediamo sta dipendenza dal tempo
    sigma_ext = 0.1
    norms_ext = np.logspace(np.log10(0.1), np.log10(10000), 1000)
    
    psf_index = np.argmin(np.abs(psf_e_means - e[energ_index]))
    sigma_1, sigma_2, sigma_3, scale, ampl_2, ampl_3 = psf_hbu.data[psf_index][0]
    
    flux_ratios = np.zeros_like(nbr_pointings)
    
    def psf_model(th2):
        gauss1 = np.exp(-th2 / (2 * sigma_1**2))
        gauss2 = ampl_2*np.exp(-th2 / (2 * sigma_2**2))
        gauss3 = ampl_3*np.exp(-th2 / (2 * sigma_3**2))
        return (gauss1+gauss2+gauss3)
        
    for i, time in enumerate(t_expos):
        
        ct = cts[i][energ_index]
        ct_err = cts_err[i][energ_index]
        
        ct_point = psf_model(th2)
        # rescaling the error
        # rescaling the psf component to match total counts of the data
        ct_point *= sum(ct)/sum(ct_point)
        ct_point_err = np.sqrt(ct_point)
    
        chi2_noext = np.sum(((ct - ct_point)**2 / (ct_err**2 + ct_point_err**2)))
    
        # taken from andrii notebook, to fit with psf better:
        step=1.01
        chi2_noext_best=1e10
        adjust=1
        while(chi2_noext<chi2_noext_best):
            chi2_noext_best=chi2_noext
            chisq0=sum((ct-step*adjust*ct_point)**2/(ct_err**2+(adjust*step*ct_point_err)**2))
            chisq1=sum((ct-adjust*ct_point/step)**2/(ct_err**2+(adjust*ct_point_err/step)**2))
            chisq_vec=np.array([chisq0,chisq1])
            chi2_noext=min(chisq_vec)
            ind=np.argmin(chisq_vec)
            if(chi2_noext<chi2_noext_best):
                if(ind==0):
                    adjust*=step
                if(ind==1):
                    adjust/=step
        ct_point=adjust*ct_point
        ct_point_err=adjust*ct_point_err
        chi2_noext= np.sum(((ct - ct_point)**2 / (ct_err**2 + ct_point_err**2)))
    
        # For each sigma_ext: add a gaussian term norm_ext*gauss(sigma_ext) convolved with the psf, to the point source model (psf only), until delta_chi2 is > 2.71.
        # With this you found norm_ext such that the fit breaks. Lets call this norm_ext_br
        # Then the upper limit to the detectable ext. emission should be given by an extended emission of the form of: norm_ext_br*gauss(sigma_ext)
        found_value = False
        for j, norm_ext in enumerate(norms_ext):
            if found_value:
                continue
            def ext_model(th2):
                sigma1_conv = np.sqrt(sigma_1**2 + sigma_ext**2)
                sigma2_conv = np.sqrt(sigma_2**2 + sigma_ext**2)
                sigma3_conv = np.sqrt(sigma_3**2 + sigma_ext**2)
                gauss1_conv = np.exp(-th2 / (2 * sigma1_conv**2))
                gauss2_conv = ampl_2 * np.exp(-th2 / (2 * sigma2_conv**2))
                gauss3_conv = ampl_3 * np.exp(-th2 / (2 * sigma3_conv**2))
                return norm_ext * (gauss1_conv + gauss2_conv + gauss3_conv)
        
            ct_ext = ext_model(th2)
            ct_point_and_ext = ct_ext+ct_point
            chi2_point_and_ext = np.sum(((ct - ct_point_and_ext)**2 / (ct_err**2 + ct_point_err**2)))             
            delta_chi2 = chi2_point_and_ext - chi2_noext
            
            if delta_chi2 > 2.71:
                flux_ratio = np.sum(ct_ext)/(np.sum(ct_point) + np.sum(ct_ext))
                flux_ratios[i] = flux_ratio
                found_value = True
                #print("converged at ")
                #print(j)
    return flux_ratios

In [ ]:
for energ_index, energ in enumerate(e):
    flux_ratios = calculate_timedep_for_given_energ(energ_index)
    def power_model(t_expos, a, b):
        return t_expos**a*b
    
    popt, _ = curve_fit(power_model, t_expos, flux_ratios)
    power_fit = power_model(t_expos, *popt)
    print("Power: ")
    print(popt[0])
    plt.plot(t_expos, power_fit)
    plt.plot(t_expos, flux_ratios, "+")
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("expos time [h]")

# la sensibilita va come 1/t.

Power: 
-0.3717955025105463
